In [2]:
# !pip install jupyter_bokeh -q

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Javascript
from io import StringIO, BytesIO
import requests
import matplotlib.pyplot as plt
import numpy as np
import json
import math
import yfinance as yf
import datetime
import datetime as dt
import aiohttp
import asyncio
import urllib3
from datetime import timedelta
from scipy import interpolate
import warnings
import panel as pn
from bokeh.plotting import figure
import concurrent.futures
from threading import Lock
import plotly.express as px
import base64
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# from bokeh.util.warnings import BokehUserWarning
# warnings.filterwarnings("ignore", category=BokehUserWarning)

pn.extension()

http = urllib3.PoolManager()


simple_check = """
<script>
if (window.innerWidth < 600) {
    document.write(`
        <div style="
            background: #ffebee;
            border-left: 4px solid #f44336;
            padding: 10px;
            margin: 10px 0;
        ">
            <strong>Note:</strong> Screen width is ${window.innerWidth}px (minimum 600px recommended)
        </div>
    `);
}
</script>
"""

# <script>
# if (window.innerWidth < 600) {
#     // Redirect langsung
#     window.location.href = "https://www.kompas.com";
# }
# </script>

display(HTML(simple_check))

# Global Styling untuk konsistensi
STYLE_W = {'description_width': '117px'}
LAYOUT_W = widgets.Layout(width='335px')
LAYOUT_BUTTONS = widgets.Layout(justify_content='center')

# Data dummy untuk demonstrasi
data_dummy = """
Date,Open,High,Low,Close,Volume
2023-01-01,100,105,99,104,100000
2023-01-02,104,106,103,105,120000
2023-01-03,105,108,104,107,110000
2023-01-04,107,109,106,108,95000
2023-01-05,108,110,107,109,130000
"""
df_data = {}
for i in range(30):
    df_data[f'SAHAM{i+1}'] = pd.read_csv(StringIO(data_dummy))

# 1. Buat widget tampilan
try:
    image_url = 'https://media.istockphoto.com/id/1022809094/vector/concentric-circles-dots-in-circular-form-vector.jpg?s=170667a&w=0&k=20&c=QBPSwryZvuP5-La1oJGzmvEm030m8VIyizAZJ3uO8Hc='
    image_data = requests.get(image_url).content
    ai_logo = pn.pane.Image(image_url, width=125, height=125)
    centered_logo = pn.Row(pn.layout.HSpacer(), ai_logo, pn.layout.HSpacer(),sizing_mode='fixed', styles={'width': '95%', 'margin-left': '10px'})
except Exception as e:
    print(f"Gagal memuat gambar dari URL. Error: {e}")
    ai_logo = pn.pane.HTML("<div style='text-align: center;'>[Gambar AI tidak dapat dimuat]</div>", width=100, height=100)

title_text = pn.pane.HTML("""<div style='text-align: center;line-height: 1.5; font-weight: bold; font-size: 17px; margin-bottom: 10px'>Hi, I'am QuantGenius<br>Superintelligent Quantitative Trading System<br>Test Me With Any Stock Data From Any Period</div>""", sizing_mode='fixed', styles={'width': '95%', 'margin-left': '10px'})

# Konten Accordion "Test Setting"
quantgenius_apikey = pn.widgets.PasswordInput(name='QuantGenius API Key', placeholder='Input QuantGenius API Key', width=500)
apikey_title = pn.widgets.StaticText(value='Anda butuh quantgenius api key untuk melakukan request realtime trade signals. Jika anda belum ada, click here https://api.quantgenius.ai. lebih detail silahkan cek quantgenius apidoc', width=500, styles={'margin-top': '0px', 'text-align': 'justify', 'color': 'gray'})
initial_equity = pn.widgets.IntInput(name='Initial Equity', value=1000000, step=100000, start=1000000, end=1000000000000, width=110)
commission = pn.widgets.FloatInput(name='Commission/trade', value=0.001, step=0.0001, start=0, end=0.01, width=110)
spread = pn.widgets.FloatInput(name='Spread/trade', value=0.001, step=0.0001, start=0, end=0.01, width=110)
interest_rate = pn.widgets.FloatInput(name='Interest Rate/year', value=0.03, step=0.01, start=0, end=0.1, width=110)
apikey_title1 = pn.widgets.StaticText(value='Anda butuh quantgenius api key untuk melakukan request realtime trade signals. Jika anda belum ada, click here https://api.quantgenius.ai. lebih detail silahkan cek quantgenius apidoc', width=500, styles={'margin-top': '0px', 'text-align': 'justify', 'color': 'gray'})
regT_margin_rate = pn.widgets.FloatInput(name='Margin Requirement', value=0.5, step=0.01, start=0, end=1, width=110)
maintenance_margin_rate = pn.widgets.FloatInput(name='Margin Maintenance', value=0.25, step=0.01, start=0, end=1, width=110)
regT_margin_rate1 = pn.widgets.FloatInput(name='Margin Requirement', value=0.5, step=0.01, start=0, end=1, width=110)
maintenance_margin_rate1 = pn.widgets.FloatInput(name='Margin Maintenance', value=0.25, step=0.01, start=0, end=1, width=110)
apikey_title3 = pn.widgets.StaticText(value='QuantGenius telah dilatih dan di uji pada margin trading dengan leverage max hanya 2x, menggunakan 30 saham portfolio. QuantGenius mampu menghasilkan return exponential dengan hanya menggunakan leverage tidak lebih dari 2x dengan resiko rendah. QuantGenius telah dilatih dan di uji pada margin trading dengan leverage max hanya 2x, menggunakan 30 saham portfolio. QuantGenius mampu menghasilkan return exponential dengan hanya menggunakan leverage tidak lebih dari 2x dengan resiko rendah.', width=550, styles={'margin-top': '0px', 'text-align': 'justify', 'color': 'gray'})

mm = []
for i in range(30):
    mr_input = pn.widgets.FloatInput(name=f'MR Stock {i+1}', value=0.5, step=0.01, start=0, end=1, width=80)
    mm_input = pn.widgets.FloatInput(name=f'MM Stock {i+1}', value=0.25, step=0.01, start=0, end=1, width=80)
    mm.append(pn.Column(mr_input, mm_input, margin=(0, 5), align='center'))
mm_container = pn.Row(*mm, width=550, height=150, scroll=True, styles={'overflow-x': 'auto', 'border': '1px solid #C0C0C0', 'padding': '5px', 'margin-left': '10px', 'margin-bottom': '10px'})
mm_title = pn.widgets.StaticText(value='Margin Requirement & Maintenance Margin (Test Using Portfolio With 30 Stocks)', width=550, styles={'margin-top': '15px', 'text-align': 'left'})
apikey_title2 = pn.widgets.StaticText(value='Anda butuh quantgenius api key untuk melakukan request realtime trade signals. Jika anda belum ada, click here https://api.quantgenius.ai. lebih detail silahkan cek quantgenius apidoc', width=500, styles={'margin-top': '0px', 'text-align': 'justify', 'color': 'gray'})
divider = pn.layout.Divider(width=100, margin=(0, 20), styles={'margin-top': '5px', 'width': '1070px'})
divider1 = pn.layout.Divider(width=100, margin=(0, 20), styles={'margin-top': '5px', 'width': '1070px'})
divider2 = pn.layout.Divider(width=100, margin=(0, 20), styles={'margin-top': '5px', 'width': '1070px'})
Himbauan_title = pn.widgets.StaticText(value='PASTIKAN SETTING ANDA SENYATA MUNGKIN AGAR HASIL SIMULASI BENAR BENAR AKURAT SESUAI TRADING NYATA. ANDA JUGA BISA MENYESUAIKAN SETTING UNTUK UJI STRESS UNTUK MELIHAT KETANGGUHAN SISTEM. PASTIKAN SETTING ANDA SENYATA MUNGKIN AGAR HASIL SIMULASI BENAR BENAR AKURAT SESUAI TRADING NYATA.', width=1070, styles={'margin-left': '20px', 'margin-bottom': '15px', 'text-align': 'justify', 'color': 'gray'})
test_setting_content = pn.Column(pn.Row(pn.Column(quantgenius_apikey, apikey_title, pn.Row(initial_equity, commission, spread, interest_rate), apikey_title1, pn.Row(regT_margin_rate, maintenance_margin_rate, regT_margin_rate1, maintenance_margin_rate1), apikey_title2, styles={'margin-left': '10px', 'margin-top': '15px', 'margin-bottom': '0px'}), pn.Column(mm_title, mm_container, apikey_title3)),divider, Himbauan_title)

CURRENT_UPLOAD_WIDGET = None
# OUTPUT_AREA = widgets.Output()
UPLOAD_CONTAINER = widgets.HBox([widgets.Label(value="", layout=widgets.Layout(width='121px'))])

UPLOADED_FILES_DATA = {} # Dictionary untuk menyimpan data berkas dari semua sesi upload

def create_dropdown(options, value, description, width='180px'):
    return widgets.Dropdown(options=options, value=value, description=description,
                            style=STYLE_W, layout=widgets.Layout(width=width))

def create_slider(value, min_val, max_val, description, is_int=False):
    WidgetType = widgets.IntSlider if is_int else widgets.FloatSlider
    return WidgetType(value=value, min=min_val, max=max_val, description=description,
                      style=STYLE_W, layout=widgets.Layout(width='320px'))

# Bagian Kontrol Atas
data_need_label = widgets.Label(value="You need 30+ stock data")
data_source_toggle = pn.widgets.Select(name='Select Data Source', options=['Local Data', 'Monte Carlo Simulation'])
# generate_button = pn.widgets.Button(name='Generate Dataset', button_type='primary', sizing_mode='fixed', styles={'width': '50%', 'margin-left': '11px'})


# Blok 1: Yahoo Finance (Tidak Berubah)
# exchange_dropdown = pn.widgets.Select(name='Select Exchange', options=list(STOCK_DATA.keys()))
# select_stock_mode = pn.widgets.Select(name='Select stock mode', options=['Manual', 'Random'])
# stock_select_multiple = pn.widgets.Select(name='Select Stocks', options= STOCK_DATA['LSE'])

# yahoo_widgets = pn.Column(exchange_dropdown, select_stock_mode, stock_select_multiple)


date_col_dropdown = widgets.Dropdown(options=['null'] + list(map(str, range(0, 10))), value='null', description='Date Column Index', style=STYLE_W, layout=widgets.Layout(width='180px'))
price_col_dropdown = widgets.Dropdown(options=['null'] + list(map(str, range(0, 10))), value='null', description='Price Column Index', style=STYLE_W, layout=widgets.Layout(width='180px'))

# Susunan widget lokal: [Upload], [Total Files Label BARU], [Date Column], [Price Column]
local_widgets = pn.Column(
    UPLOAD_CONTAINER)

WIDGET_MAP = {
    'Local Data': local_widgets,
    'Monte Carlo Simulation': local_widgets
}

def create_upload_widget():
    """Membuat widget FileUpload baru dan menempatkannya di kontainer."""
    global CURRENT_UPLOAD_WIDGET

    new_upload_widget = widgets.FileUpload(
        accept='.csv,.txt', multiple=True, button_style='primary', description='Upload your stock data', layout=widgets.Layout(width='210px')
    )
    new_upload_widget.observe(on_local_data_upload, names='value')

    CURRENT_UPLOAD_WIDGET = new_upload_widget
    UPLOAD_CONTAINER.children = (UPLOAD_CONTAINER.children[0], new_upload_widget)

def on_local_data_upload(change):
    """Observer untuk FileUpload. Menyimpan data, memproses log, lalu me-reset widget."""
    global UPLOADED_FILES_DATA

    UPLOADED_FILES_DATA.update(change['new'])
    create_upload_widget()

def update_ui(change):
    """Menyembunyikan/menampilkan widget berdasarkan pilihan sumber data."""
    selected_source = change['new']
    for widget in WIDGET_MAP.values():
        widget.layout.display = 'none'

    WIDGET_MAP[selected_source].layout.display = 'block'
    # plot_button.layout.display = 'block' if selected_source == 'Monte Carlo Simulation' else 'none'

    if selected_source == 'Local Data':
        # Pastikan label hitungan diperbarui saat panel diaktifkan
        # update_total_files_label()
        if CURRENT_UPLOAD_WIDGET is None or CURRENT_UPLOAD_WIDGET not in UPLOAD_CONTAINER.children:
            create_upload_widget()

def clean_data_simple(df):
    """
    Membersihkan data harga dengan interpolasi linear sederhana.
    Versi yang lebih robust untuk menghindari masalah indexing.
    """
    column="Close"
    threshold_drop=-0.5
    threshold_jump=1.0
    try:
        # Return langsung jika kolom tidak ada atau data pendek
        if column not in df.columns or len(df) < 3:
            return df.copy()

        cleaned = df.copy()
        prices = cleaned[column]

        # Reset index sementara untuk menghindari masalah indexing
        original_index = cleaned.index
        cleaned_reset = cleaned.reset_index(drop=True)
        prices_reset = cleaned_reset[column]

        # Hitung persentase perubahan
        pct_change = prices_reset.pct_change()

        # Cari indices dimana perubahan melebihi threshold
        drop_mask = (pct_change < threshold_drop).to_numpy()
        jump_mask = (pct_change > threshold_jump).to_numpy()
        anomaly_mask = drop_mask | jump_mask

        # Dapatkan indices sebagai array numpy
        anomaly_indices = np.where(anomaly_mask)[0]

        # Iterasi melalui anomaly indices
        for idx in anomaly_indices:
            # Pastikan kita tidak di awal atau akhir data
            if 0 < idx < len(prices_reset) - 1:
                prev_val = prices_reset.iloc[idx-1]
                next_val = prices_reset.iloc[idx+1]

                # Skip jika nilai sebelumnya atau sesudahnya NaN
                if not (np.isnan(prev_val) or np.isnan(next_val)):
                    # Interpolasi linear
                    interpolated_value = (prev_val + next_val) / 2
                    cleaned_reset.at[idx, column] = interpolated_value

        # Kembalikan index asli
        cleaned_reset.index = original_index
        return cleaned_reset

    except Exception as e:
        print(f"Error dalam pembersihan data: {e}")
        import traceback
        traceback.print_exc()
        return df.copy()

def html_head_tail_single_table(df, n):
    """
    Menampilkan head dan tail DataFrame dengan lebar kolom otomatis (sesuai teks)
    menggunakan Pandas Styler.
    """

    # 1. Persiapan Data
    df.index = df.index.astype(str)
    head_df = df.head(n)
    tail_df = df.tail(n)
    separator_row = pd.DataFrame([['...'] * len(df.columns)], columns=df.columns, index=['...'])
    combined_df = pd.concat([head_df, separator_row, tail_df])

test_data = pd.DataFrame()
tanggal = np.array([])
dataHarga = np.array([])

def resetdata_clicked(event=None):
    global dataHarga, test_data, tanggal
    test_data = []
    tanggal = []
    dataHarga = []
    display_pane.visible = False
    display_pane.object = pd.DataFrame()
    create_dataset_button.disabled = False
    reset_button.disabled = True

def create_dataset(event=None):

    create_dataset_button.disabled = True

    """Mencetak ringkasan parameter yang dipilih."""
    # with OUTPUT_AREA:
    global dataHarga, test_data, tanggal

    source = data_source_toggle.value
    # generate_button.disabled = True
    start_button.disabled = True
    error_pane.visible = False

    tickers = ("AAME","GABC","ABEO","NEON","ADBE","ADI","ADP","ADSK","HLIT","AGYS","CWCO","AIRT","ALCO","ALOT","GIII","CASH","AMD","AMGN","IIIN","AMWD","MATW","APOG","ARKR","AROW","ARTW","ASRV","ASTE","INCY","AAPL","ATRO")
    # tickers = ("BX", "CLPR", "KODK", "VISL", "OIS", "SMBC", "CAC", "VKI", "MAN", "ETJ", "NRIM", "CPS", "BGFV", "BMRA", "MSM", "WTM", "SIG", "INO", "IDXX", "PHD", "RMD", "RFIL", "FBRX", "PLUS", "MLM", "ACHV", "ACR", "NTZ", "XRAY", "GNTY")
    # tickers = ("001065.ks","001067.ks","0011.hk","0012.hk","001230.ks","001440.ks","001465.ks","001520.ks","001525.ks","001550.ks","0016.hk","001620.ks","001680.ks","0017.hk","001740.ks","001790.ks","001800.ks","0019.hk","0021.hk","0023.hk","002350.ks","002460.ks","0027.hk","002790.ks","0028.hk","003075.ks","0033.hk","003470.ks","0035.hk","003545.ks")
    # tickers = stock_select_multiple.value

    # Unduh semua data sekaligus
    portfolio_data_all = yf.download(tickers, period="max", auto_adjust=False, progress=False, group_by='ticker')

    portfolioTicker = []
    portfolioData = []

    for ticker in tickers:
        try:
            # Ambil data untuk ticker tertentu dari hasil download batch
            if ticker in portfolio_data_all.columns.get_level_values(0):
                tickerSelect_data = portfolio_data_all[ticker].copy()

                # Konversi index ke format tanggal
                tickerSelect_data.index = pd.to_datetime(tickerSelect_data.index).date

                # Hapus baris yang semua nilainya NaN
                tickerSelect_data = tickerSelect_data.dropna(how='all')

                # Cek jumlah minimal data
                if len(tickerSelect_data) > 1000:
                    portfolioTicker.append(ticker)
                    portfolioData.append(tickerSelect_data)

        except Exception as e:
            print(f"Error processing {ticker}: {e}")
            continue

    # Find common date range
    if portfolioData:
        testStartdate = max([data.index.min() for data in portfolioData])
        testEnddate = min([data.index.max() for data in portfolioData])

        # Create common date range (hanya hari kerja)
        common_dates = pd.date_range(start=testStartdate, end=testEnddate, freq='B').date  # 'B' untuk Business days

        # Filter out weekends from each portfolio data
        portfolioData_weekdays = []
        for data in portfolioData:
            # Konversi index ke datetime untuk filtering
            data_index_dt = pd.to_datetime(data.index)
            # Filter hanya hari kerja (Senin-Jumat)
            weekday_mask = data_index_dt.dayofweek < 5  # 0=Senin, 4=Jumat, 5=Sabtu, 6=Minggu
            data_weekdays = data[weekday_mask]
            portfolioData_weekdays.append(data_weekdays)

        # Combine all data
        combined_data = pd.DataFrame(index=common_dates)

        for i, (ticker, data) in enumerate(zip(portfolioTicker, portfolioData_weekdays)):
            combined_data[ticker] = data['Close'].reindex(common_dates)

        # Handle missing values
        combined_data.ffill(inplace=True)
        combined_data.bfill(inplace=True)

        # Final data - pastikan hanya hari kerja yang tersisa
        test_data = combined_data.dropna()  # Remove any remaining NaN

        # Verifikasi bahwa tidak ada data weekend
        final_dates = pd.to_datetime(test_data.index)
        weekend_mask = final_dates.dayofweek >= 5
        if weekend_mask.any():
            print("Peringatan: Masih terdapat data weekend, melakukan pembersihan tambahan...")
            test_data = test_data[~weekend_mask]

        dataHarga = test_data.to_numpy()
        tanggal = test_data.index.to_numpy()

        display_data = test_data.copy()
        display_data.index = display_data.index.astype(str)

        # PERBAIKAN 2: Ambil head dan tail
        head_df = display_data.head(5)
        tail_df = display_data.tail(5)

        # PERBAIKAN 3: Buat separator row dengan cara yang benar
        separator_data = {col: ['...'] for col in display_data.columns}
        separator_row = pd.DataFrame(separator_data, index=['...'])

        # PERBAIKAN 4: Gabungkan dengan concat
        combined_df = pd.concat([head_df, separator_row, tail_df])

        # Tampilkan hasil
        display_pane.object = combined_df
        display_pane.visible = True  # Tampilkan pane
        error_pane.visible = False
        reset_button.disabled = False
        start_button.disabled = False

select_dataset_title = pn.widgets.StaticText(value='Select Dataset', styles={'margin-left': '15px', 'margin-top': '15px', 'margin-bottom': '0px'})
select_dataset = pn.widgets.RadioBoxGroup(name='select_dataset', options=['Montecarlo Simulation', 'Local Data'], inline=True, styles={'margin-left': '15px', 'margin-top': '15px', 'margin-bottom': '0px'})
select_simulation_method = pn.widgets.Select(name='Select Simulation Method', options=['Method1', 'Method2', 'Method3'], styles={'margin-left': '15px', 'margin-top': '5px', 'margin-bottom': '0px'})
simulation_method_title = pn.widgets.StaticText(value=select_simulation_method.options[0]+' Simulation Setting :', styles={'margin-left': '15px', 'margin-top': '15px', 'margin-bottom': '0px'})
display_pane = pn.pane.DataFrame(pd.DataFrame(), width=750, height=200, visible=False)
create_dataset_button = pn.widgets.Button(name=f'Create Dataset', button_type='primary', styles={'margin-left': '15px', 'margin-top': '10px', 'margin-bottom': '10px'})
reset_button = pn.widgets.Button(name='Reset Dataset', button_type='danger', disabled=True, styles={'margin-left': '15px', 'margin-top': '10px', 'margin-bottom': '10px'})
dataset_title = pn.widgets.StaticText(value='QuantGenius telah dilatih dan di uji pada margin trading dengan leverage max hanya 2x, menggunakan 30 saham portfolio. QuantGenius mampu menghasilkan return exponential dengan hanya menggunakan leverage tidak lebih dari 2x dengan resiko rendah. QuantGenius telah dilatih dan di uji pada margin trading dengan leverage max hanya 2x, menggunakan 30 saham portfolio. QuantGenius mampu menghasilkan return exponential dengan hanya menggunakan leverage tidak lebih dari 2x dengan resiko rendah.', width=1070, styles={'margin-top': '20px', 'margin-left': '20px', 'text-align': 'justify', 'color': 'gray'})
Himbauan_title2 = pn.widgets.StaticText(value='PASTIKAN SETTING ANDA SENYATA MUNGKIN AGAR HASIL SIMULASI BENAR BENAR AKURAT SESUAI TRADING NYATA. ANDA JUGA BISA MENYESUAIKAN SETTING UNTUK UJI STRESS UNTUK MELIHAT KETANGGUHAN SISTEM. PASTIKAN SETTING ANDA SENYATA MUNGKIN AGAR HASIL SIMULASI BENAR BENAR AKURAT SESUAI TRADING NYATA.', width=1070, styles={'margin-left': '20px', 'margin-bottom': '15px', 'text-align': 'justify', 'color': 'gray'})

# def toggle_display(event):
#   if button.name == 'Create Dataset':
#       create_dataset()
#       # Pastikan test_data ada dan tidak kosong
#       if test_data is not None and not test_data.empty:
#           # PERBAIKAN 1: Gunakan test_data.index langsung, tidak perlu df.index
#           # Konversi index ke string untuk display yang lebih baik
#           display_data = test_data.copy()
#           display_data.index = display_data.index.astype(str)

#           # PERBAIKAN 2: Ambil head dan tail
#           head_df = display_data.head(5)
#           tail_df = display_data.tail(5)

#           # PERBAIKAN 3: Buat separator row dengan cara yang benar
#           separator_data = {col: ['...'] for col in display_data.columns}
#           separator_row = pd.DataFrame(separator_data, index=['...'])

#           # PERBAIKAN 4: Gabungkan dengan concat
#           combined_df = pd.concat([head_df, separator_row, tail_df])

#           # Tampilkan hasil
#           display_pane.object = combined_df
#           display_pane.visible = True  # Tampilkan pane
#           button.name = 'Reset Dataset'
#           button.button_type = 'danger'
#       else:
#           # Jika tidak ada data, tampilkan pesan error
#           error_df = pd.DataFrame({'Error': ['No data available. Please check your data source.']})
#           display_pane.object = error_df
#           display_pane.visible = True
  # else:
  #     # Reset ke DataFrame kosong
  #     resetdata_clicked
  #     display_pane.visible = False
  #     display_pane.object = pd.DataFrame()
  #     button.name = 'Create Dataset'
  #     button.button_type = 'primary'



  # return pn.Row(pn.Column(pn.Row(select_dataset_title, select_dataset), select_simulation_method, simulation_method_title, button),display_pane)

# select_dataset.param.watch(update_ui, 'value')
reset_button.on_click(resetdata_clicked)

create_upload_widget()
create_dataset_button.on_click(create_dataset)


dataset_content = pn.Column(dataset_title, divider1, pn.Row(pn.Column(pn.Row(select_dataset_title, select_dataset), select_simulation_method, simulation_method_title, pn.Row(create_dataset_button, reset_button)),display_pane), divider2, Himbauan_title2)
# dataset_content = pn.Column(
#         data_source_toggle,
#         table_container,
#         pn.Row(generate_button,reset_button), styles={'margin-left': '10px', 'margin-top': '15px', 'margin-bottom': '15px'})

#=======================================================================================================================================================================

image = requests.get("https://raw.githubusercontent.com/guangyoung/test2/refs/heads/main/360_F_1394167414_FkutDkFwLRQETxtp7PdrmBWBYMGoezIY.jpg").content
imge = widgets.Image(
    value=image,
    format='jpg',
    width=745
)

image2 = requests.get("https://raw.githubusercontent.com/guangyoung/test2/refs/heads/main/360_F_1394167414_FkutDkFwLRQETxtp7PdrmBWBYMGoezIY_copy.jpg").content
imge2 = widgets.Image(
    value=image2,
    format='jpg',
    width=745
)

p1 = figure(width=300, height=300, name='Scatter', margin=5)
p1.scatter([0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 2, 1, 0])

main_accordion = pn.Accordion(
    ('Test Setting', test_setting_content),
    ('Dataset', dataset_content),
    # toggle=True,
    # sizing_mode='stretch_width',
    styles={'width': '95%', 'margin-left': '6px'}
)

start_button = pn.widgets.Button(name='▶️ Start Trade Simulation With Real-Time QuantGenius Trade Signals', button_type='success', styles={'width': '95%', 'margin-left': '10px'})

# progress = pn.widgets.Progress(name='Progress dengan Animasi', value=0, bar_color='success', visible=True, sizing_mode='stretch_width', styles={'width': '95%', 'margin-left': '11px'})
progress_pane = pn.pane.Markdown("Proses sedang berjalan...", visible=False, styles={'margin-top': '-10px'})

github_button = pn.widgets.Button(
    name='📊 Github Profile',
    button_type='primary',
    width=170,
    styles={
        'margin': '5px',
        'font-size': '12px',
        'cursor': 'pointer'
    }
)
github_button.js_on_click(code="window.open('https://www.github.com', '_blank')")

# About QuantGenius Button
about_button = pn.widgets.Button(
    name='📚 About QuantGenius',
    button_type='primary',
    width=170,
    styles={
        'margin': '5px',
        'font-size': '12px',
        'cursor': 'pointer'
    }
)
about_button.js_on_click(code="window.open('https://www.quantgenius.ai', '_blank')")

# Google Colab Button
colab_button = pn.widgets.Button(
    name='⚡ Open in Google Colab',
    button_type='primary',
    width=170,
    styles={
        'margin': '5px',
        'font-size': '12px',
        'cursor': 'pointer'
    }
)
colab_button.js_on_click(code="window.open('https://colab.research.google.com', '_blank')")

# API Docs Button
api_button = pn.widgets.Button(
    name='🔗 QuantGenius API Doc',
    button_type='primary',
    width=170,
    styles={
        'margin': '5px',
        'font-size': '12px',
        'cursor': 'pointer'
    }
)
api_button.js_on_click(code="window.open('https://editor.swagger.io/?_gl=1*1ua8uzl*_gcl_au*NDAxMzAyMjg5LjE3NTc1OTU0NTk.', '_blank')")

# Centered layout
centered_badges = pn.Row(
    pn.layout.HSpacer(),
    github_button,
    about_button,
    colab_button,
    api_button,
    pn.layout.HSpacer(),
    sizing_mode='fixed', styles={'width': '95%', 'margin-left': '10px'}
)

# =================================================================================
## FUNGSI PERHITUNGAN METRIK
# =================================================================================

def calculate_metrics(equity_array, initial_equity_value, risk_free_rate=0.02):
    """Menghitung Total Return, Max Drawdown, Sharpe Ratio, dan Sortino Ratio."""

    if len(equity_array) == 0:
        return {"Total Return": np.nan, "Max Drawdown": np.nan, "Sharpe Ratio": np.nan, "Sortino Ratio": np.nan}

    total_return = (equity_array[-1] - initial_equity_value) / initial_equity_value
    daily_returns = pd.Series(equity_array).pct_change().dropna().values

    # Max Drawdown
    cumulative_return = np.insert(equity_array / initial_equity_value, 0, 1.0)
    peak = np.maximum.accumulate(cumulative_return)
    drawdown = (peak - cumulative_return) / peak
    max_drawdown = np.max(drawdown)

    if len(daily_returns) < 1:
        return {"Total Return": total_return, "Max Drawdown": max_drawdown, "Sharpe Ratio": np.nan, "Sortino Ratio": np.nan}

    annual_risk_free_rate = risk_free_rate / 252
    avg_daily_return = np.mean(daily_returns)
    std_daily_return = np.std(daily_returns)

    # Sharpe Ratio
    if std_daily_return == 0:
        sharpe_ratio = np.inf if avg_daily_return > annual_risk_free_rate else np.nan
    else:
        sharpe_ratio = (avg_daily_return - annual_risk_free_rate) / std_daily_return * np.sqrt(252)

    # Sortino Ratio
    downside_returns = daily_returns[daily_returns < 0]
    if len(downside_returns) == 0:
        sortino_ratio = np.inf if avg_daily_return > annual_risk_free_rate else np.nan
    else:
        downside_deviation = np.std(downside_returns)
        if downside_deviation == 0:
            sortino_ratio = np.inf if avg_daily_return > annual_risk_free_rate else np.nan
        else:
            sortino_ratio = (avg_daily_return - annual_risk_free_rate) / downside_deviation * np.sqrt(252)

    return {"Total Return": total_return, "Max Drawdown": max_drawdown, "Sharpe Ratio": sharpe_ratio, "Sortino Ratio": sortino_ratio}

def open_new_tab_base64(fig_pie):
    plot_html = fig_pie.to_html(include_plotlyjs='cdn')

    plot_html_encoded = base64.b64encode(plot_html.encode()).decode()

    js = f"""
    <script>
        try {{
            var plotHTML = atob("{plot_html_encoded}");
            var newWindow = window.open();
            if (newWindow) {{
                newWindow.document.write(plotHTML);
                newWindow.document.title = "Pie Chart Interaktif";
                newWindow.document.close();
            }} else {{
                alert("Popup diblokir! Izinkan popup untuk situs ini.");
            }}
        }} catch (error) {{
            console.error("Error:", error);
            alert("Terjadi error: " + error.message);
        }}
    </script>
    """
    display(HTML(js))

def display_full_report_new_tab(fig, daftar_saham, df_perf):
    # 1. Konversi Grafik Plotly ke HTML (hanya bagian div dan script)
    # Kita tidak menggunakan full_html=True agar bisa kita bungkus sendiri
    plot_div = fig.to_html(full_html=False, include_plotlyjs='cdn')

    # HTML Daftar Saham (Scrollbox)
    saham_items = "".join([f'<div style="background:white; padding:8px; border:1px solid #eee; border-radius:4px; font-size:12px;"><b>{s.split(" - ")[0]}</b> - {s.split(" - ")[1]} - {s.split(" - ")[2]}</div>' for s in daftar_saham])

    # portfolio_section = f"""
    # <div style="width: 1200px; background:#fff; padding:20px; border-radius:10px; margin-bottom:20px; border: 1px solid #ddd;">
    #     <h3 style="margin-top:0; font-family:sans-serif; color:#2c3e50;">📋 Portfolio Universe</h3>
    #     <div style="max-height: 150px; overflow-y: auto; padding: 10px; background: #f9f9f9; border-radius: 5px; display:grid; grid-template-columns: repeat(auto-fill, minmax(220px, 1fr)); gap:10px; font-family:sans-serif;">
    #         {saham_items}
    #     </div>
    # </div>
    # """

    # 3. Susun HTML Tabel Performa
    # table_section = f"""
    # <div style="width: 1200px; margin-top:30px; font-family:sans-serif;">
    #     <h2 style="color:#2c3e50;">📊 Perbandingan Performa Strategi</h2>
    #     <style>
    #         .perf-table {{ width:100%; border-collapse:collapse; margin-top:10px; background:white; }}
    #         .perf-table th {{ background:#2c3e50; color:white; padding:12px; text-align:center; }}
    #         .perf-table td {{ border:1px solid #dee2e6; padding:10px; text-align:center; }}
    #         .perf-table tr:nth-child(even) {{ background:#f2f2f2; }}
    #     </style>
    #     {df_perf.to_html(index=False, classes='perf-table')}
    # </div>
    # """

    # 4. Gabungkan Semuanya menjadi satu Dokumen HTML utuh
    full_document = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Quantxi System Report</title>
        <style>
            body {{ background-color: #f4f7f6; padding: 40px; font-family: sans-serif; }}
            .container {{ max-width: 1200px; margin: auto; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div style="width: 1200px; background:#fff; padding:20px; border-radius:10px; margin-bottom:20px; border: 1px solid #ddd;">
                <h3 style="margin:0; font-family:sans-serif; color:#2c3e50;">Portfolio<span style="margin-left: 750px; font-family:sans-serif; color:#2c3e50">
                    Period of Data : 01/01/2021 - 01/01/2026 </span></h3>
                <div style="max-height: 150px; overflow-y: auto; padding: 10px; background: #f9f9f9; border-radius: 5px; display:grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap:10px; font-family:sans-serif;">
                    {saham_items}
                </div>
            </div>
            <div style="width: 1200px; background:white; padding:15px; border-radius:10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
                {plot_div}
            </div>
            <div style="width: 1200px; margin-top:30px; font-family:sans-serif;">
                <h2 style="color:#2c3e50;">Performance Comparison</h2>
                <style>
                    .perf-table {{ width:100%; border-collapse:collapse; margin-top:10px; background:white; }}
                    .perf-table th {{ background:#2c3e50; color:white; padding:12px; text-align:center; }}
                    .perf-table td {{ border:1px solid #dee2e6; padding:10px; text-align:center; }}
                    .perf-table tr:nth-child(even) {{ background:#f2f2f2; }}
                </style>
                {df_perf.to_html(index=False, classes='perf-table')}
            </div>
        </div>
    </body>
    </html>
    """

    # 5. Encode dan Kirim ke Jendela Baru via JavaScript
    full_html_encoded = base64.b64encode(full_document.encode()).decode()

    js_code = f"""
    <script>
        try {{
            var fullHTML = atob("{full_html_encoded}");
            var newWindow = window.open();
            if (newWindow) {{
                newWindow.document.open();
                newWindow.document.write(fullHTML);
                newWindow.document.title = "Laporan Lengkap Quantxi";
                newWindow.document.close();
            }} else {{
                alert("Popup diblokir! Harap izinkan popup untuk melihat laporan.");
            }}
        }} catch (error) {{
            console.error("Error:", error);
            alert("Gagal membuka laporan: " + error.message);
        }}
    </script>
    """
    display(HTML(js_code))
# =================================================================================
## FUNGSI SIMULASI YANG DIPERBAIKI (LEBIH CEPAT)
# =================================================================================

def start_simulation(test_data, dataHarga, tanggal):

    reset = http.request("DELETE", "https://www.quantgenius.ai/api/resetdata", headers = {"X-QUANTGENIUS-APIKEY":"2NV4yE070VhcarVhZRJN7u3ZWP8zeq9oCsely819"})
    # print(reset.json())

    total_iterations = 300
    # total_iterations = len(test_data)
    start_button.disabled = True
    reset_button.disabled = True
    progress_pane.visible = True

    # 2. Inisialisasi Variabel
    stock_price = np.zeros(30, dtype='d')
    stock_price_sebelum = np.zeros(30, dtype='d')
    stock_position_size = np.zeros(30, dtype='d')

    initial_stockPrice = np.array(dataHarga[0], dtype='d')
    initial_eq = initial_equity.value
    cash_balance = initial_eq

    # GANTI np.array APPEND DENGAN LIST APPEND (LEBIH CEPAT)
    quantxi_equity_list = []
    buyhold_equity_list = []
    market_value_list = []

    previous_signal_hashcode = "0000000000000000000000000000000000000000000000000000000000000000"
    dataIdx = 0
    accrued_interest = 0
    min_excessLiquidity = initial_eq

    stock_price_sebelum = dataHarga[dataIdx]

    # 3. Loop Simulasi
    while dataIdx < total_iterations:

        stock_price = dataHarga[dataIdx]

        # Logika Bunga
        if cash_balance < 0:
            daily_Interest = cash_balance * (interest_rate.value / 360) * (tanggal[dataIdx] - tanggal[dataIdx-1]).days
            accrued_interest += daily_Interest

        if tanggal[dataIdx].day == 1:
            cash_balance += accrued_interest
            accrued_interest = 0

        market_value = np.sum(stock_price * stock_position_size)
        equity_with_loanValue = cash_balance + market_value + accrued_interest

        portfolio_data = [{"price": stock_price[i], "position_size": stock_position_size[i]} for i in range(30)]
        data_input = {
            "prev_signal_hashcode": previous_signal_hashcode,
            "equity_balance": equity_with_loanValue,
            "portfolio_data": portfolio_data
        }

        response = http.request("POST", "https://www.quantgenius.ai/api/adddata", headers = {"X-QUANTGENIUS-APIKEY":"2NV4yE070VhcarVhZRJN7u3ZWP8zeq9oCsely819", "Content-Type": 'application/json'}, json = data_input)

        # signal_output = response.json()["data"]

        response_data = response.json()
        signal_output = response_data.get("data", {
            "signal_hashcode": previous_signal_hashcode,
            "trade_signal": [{"action": "HOLD", "quantity": 0} for _ in range(30)]
        })

        # previous_signal_hashcode = signal_output["signal_hashcode"]
        previous_signal_hashcode = signal_output.get("signal_hashcode", previous_signal_hashcode)

        # ----------------------------------------------------------------------------------
        # TRADE TRANSACTION (Dioptimasi dengan Vektorisasi NumPy)
        # ----------------------------------------------------------------------------------
        maintenance_margin_req = market_value * maintenance_margin_rate.value
        excess_liquidity = equity_with_loanValue - maintenance_margin_req
        regT_margin_req = market_value * regT_margin_rate.value
        excess_equity = equity_with_loanValue - regT_margin_req

        if excess_liquidity > 0:
            # Inisialisasi variabel
            estimate_imr = 0
            estimate_comm = 0
            filled_percentage = 0

            # Loop pertama untuk menghitung estimate_imr dan estimate_comm
            for i in range(30):
                action = signal_output["trade_signal"][i]["action"]
                quantity = float(signal_output["trade_signal"][i]["quantity"])
                price = stock_price[i]

                if action == "BUY":
                    estimate_imr += (quantity * price) * regT_margin_rate.value
                    estimate_comm += abs(quantity) * price * (commission.value + spread.value)
                elif action == "SELL":
                    estimate_imr -= (quantity * price) * regT_margin_rate.value
                    estimate_comm += abs(quantity) * price * (commission.value + spread.value)
                else:
                    # Tidak melakukan apa-apa untuk action selain BUY/SELL
                    pass

            # Menghitung filled_percentage
            total_estimate = estimate_imr + estimate_comm

            if total_estimate <= 0:
                filled_percentage = 1
            elif excess_equity <= 0:
                filled_percentage = 0
            elif excess_equity > total_estimate:
                filled_percentage = 1
            else:
                filled_percentage = excess_equity / total_estimate

            # Inisialisasi array
            filledOrder = []
            filledPrice = []
            tradeValue = []
            commission_arr = []

            # Loop kedua untuk menghitung order yang terisi
            for i in range(30):
                action = signal_output["trade_signal"][i]["action"]
                quantity = float(signal_output["trade_signal"][i]["quantity"])
                price = stock_price[i]

                if action == "BUY":
                    filledOrder.append(quantity * filled_percentage)
                    filledPrice.append(price)
                elif action == "SELL":
                    filledOrder.append(-(quantity * filled_percentage))
                    filledPrice.append(price)
                else:
                    filledOrder.append(0)
                    filledPrice.append(0)

                # Hitung trade value dan commission
                trade_value_item = filledOrder[i] * filledPrice[i]
                commission_item = abs(filledOrder[i]) * filledPrice[i] * (commission.value + spread.value)

                tradeValue.append(trade_value_item)
                commission_arr.append(commission_item)

            # Hitung total
            total_trade_value = sum(tradeValue)
            total_commission = sum(commission_arr)

            # Update Cash dan Posisi
            cash_balance -= total_trade_value + total_commission
            stock_position_size += filledOrder

        stock_price_sebelum = dataHarga[dataIdx]

        quantxi_equity_list.append(equity_with_loanValue)

        # Buy & Hold Equity (Vektorisasi Cepat)
        bh_position = initial_eq / 30 / initial_stockPrice
        buyhold_equity = np.sum(bh_position * stock_price)
        buyhold_equity_list.append(buyhold_equity)

        market_value_list.append(market_value)

        if excess_liquidity < min_excessLiquidity:
            min_excessLiquidity = excess_liquidity

        dataIdx += 1

        progress_percent = (dataIdx / total_iterations) * 100

        # Format string progress
        progress_string = (
            f"**Proses sedang berjalan...** "
            f"Total task berjalan **{dataIdx}/{total_iterations}**, "
            f"progress **{progress_percent:.2f}%**"
        )

        progress_pane.object = progress_string

    # print("PROSES SELESAI")
    progress_pane.visible = False
    # data = {
    #     'SYSTEM': ['QUANTGENIUS', 'BUY AND HOLD'],
    #     'TOTAL RETURN': ['10,000,000.00%', '10,000.00%'],
    #     'CAGR': ['100.00%', '5.00%'],
    #     'MAXDD ABSOLUT': ['65.00%', '55.00%'],
    #     'MAXDD RELATIF': ['25.00%', '55.00%'],
    #     'CALMAR ABSOLUT': ['1.54', '0.09'],
    #     'CALMAR RELATIF': ['4.00', '0.09'],
    #     'SHARPE RATIO': ['0.80', '0.60'],
    #     'SORTINO RATIO': ['1.10', '0.70']
    # }

    # df = pd.DataFrame(data)

    # # 2. Mengubah nama kolom dengan HTML line breaks untuk header 2 baris
    # df.columns = [
    #     'SYSTEM',
    #     'TOTAL<br>RETURN',
    #     'CAGR',
    #     'MAX DD<br>ABSOLUT',
    #     'MAX DD<br>RELATIF',
    #     'CALMAR<br>ABSOLUT',
    #     'CALMAR<br>RELATIF',
    #     'SHARPE<br>RATIO',
    #     'SORTINO<br>RATIO'
    # ]

    # Konversi List ke Array NumPy
    quantxi_equity_array = np.array(quantxi_equity_list, dtype='d')
    buyhold_equity_array = np.array(buyhold_equity_list, dtype='d')
    market_value_array = np.array(market_value_list, dtype='d')

    # Hitung Metrik
    metrics_qg = calculate_metrics(quantxi_equity_array, initial_eq)
    metrics_bh = calculate_metrics(buyhold_equity_array, initial_eq)

    # clear_output(wait=True) # Hapus progress bar final

    # --- KONSTANTA ---
    WINDOW = 252*10
    ANNUAL_TRADING_DAYS = 252
    RISK_FREE_RATE = 0.0 # Asumsi Rf harian = 0

    # --- VISUALISASI GRAFIK ---
    dates = pd.to_datetime(tanggal[:total_iterations])
    qg_returns = (quantxi_equity_array - initial_eq) / initial_eq
    bh_returns = (buyhold_equity_array - initial_eq) / initial_eq

    # Ubah subplot menjadi 8 baris
    # fig, (ax1, ax3, ax8) = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    # Tambahkan jarak vertikal antar plot untuk memberi ruang pada Catatan
    # plt.subplots_adjust(hspace=8.5)

    # ----------------------------------------------------------------------------------
    # Plot 1: Return Kumulatif
    # ----------------------------------------------------------------------------------
    # ax1.plot(dates, qg_returns * 100, label='QuantGenius', color='blue')
    # ax1.plot(dates, bh_returns * 100, label='Buy & Hold', color='red')
    # ax1.set_title('Perbandingan Total Return Kumulatif', fontsize=14)
    # ax1.set_ylabel('Return (%)', fontsize=10)
    # ax1.tick_params(axis='x', rotation=45)
    # ax1.legend(loc='upper left')
    # ax1.grid(True, linestyle='--', alpha=0.6)
    # ax1.text(0.0, -0.2,
    #         'Catatan: Menunjukkan pertumbuhan modal dari waktu ke waktu. QG (biru) dibandingkan B&H (merah).\nMenunjukkan pertumbuhan modal dari waktu ke waktu. QG (biru) dibandingkan B&H (merah).\nMenunjukkan pertumbuhan modal dari waktu ke waktu. QG (biru) dibandingkan B&H (merah).',
    #         transform=ax1.transAxes, fontsize=9, color='black', ha='left')

    # ----------------------------------------------------------------------------------
    # --- PERHITUNGAN VOLATILITAS, RETURNS & RASIO ---
    # ----------------------------------------------------------------------------------

    # Log Returns Harian
    qg_log_returns = pd.Series(np.diff(np.log(quantxi_equity_array)))
    bh_log_returns = pd.Series(np.diff(np.log(buyhold_equity_array)))

    # Volatilitas Bergulir Harian (StDev)
    qg_rolling_stdev = qg_log_returns.rolling(window=WINDOW).std()
    bh_rolling_stdev = bh_log_returns.rolling(window=WINDOW).std()

    # 1. Rolling Annualized Volatility (untuk Plot 7)
    qg_rolling_volatility = qg_rolling_stdev * np.sqrt(ANNUAL_TRADING_DAYS)
    bh_rolling_volatility = bh_rolling_stdev * np.sqrt(ANNUAL_TRADING_DAYS)

    # 2. Rolling Annualized Mean Return
    qg_rolling_mean_ret = qg_log_returns.rolling(window=WINDOW).mean() * ANNUAL_TRADING_DAYS
    bh_rolling_mean_ret = bh_log_returns.rolling(window=WINDOW).mean() * ANNUAL_TRADING_DAYS

    # 3. Rolling Sharpe Ratio (untuk Plot 5)
    qg_rolling_sharpe = (qg_rolling_mean_ret - RISK_FREE_RATE) / qg_rolling_volatility
    bh_rolling_sharpe = (bh_rolling_mean_ret - RISK_FREE_RATE) / bh_rolling_volatility

    qg_rolling_sharpe = qg_rolling_sharpe.dropna()
    bh_rolling_sharpe = bh_rolling_sharpe.dropna()

    print(((qg_log_returns.mean() * ANNUAL_TRADING_DAYS) - RISK_FREE_RATE) / (qg_log_returns.std() * np.sqrt(ANNUAL_TRADING_DAYS)))
    print(((bh_log_returns.mean() * ANNUAL_TRADING_DAYS) - RISK_FREE_RATE) / (bh_log_returns.std() * np.sqrt(ANNUAL_TRADING_DAYS)))

    # 4. Rolling Sortino Ratio (untuk Plot 6)
    qg_downside_returns = qg_log_returns[qg_log_returns < 0]
    bh_downside_returns = bh_log_returns[bh_log_returns < 0]
    qg_rolling_downside_dev = qg_downside_returns.rolling(window=WINDOW).std()
    bh_rolling_downside_dev = bh_downside_returns.rolling(window=WINDOW).std()
    qg_annualized_downside_dev = qg_rolling_downside_dev * np.sqrt(ANNUAL_TRADING_DAYS)
    bh_annualized_downside_dev = bh_rolling_downside_dev * np.sqrt(ANNUAL_TRADING_DAYS)
    qg_rolling_sortino = qg_rolling_mean_ret / qg_annualized_downside_dev
    bh_rolling_sortino = bh_rolling_mean_ret / bh_annualized_downside_dev

    # Rasio Keuangan
    epsilon = 1000000
    # Menggunakan market_value_safe untuk menghindari pembagian nol yang menyebabkan NaN pada Plot 3 & 4
    market_value_safe = np.where(market_value_array == 0, epsilon, market_value_array)

    qg_equity_to_market_value = quantxi_equity_array / market_value_safe
    qg_leverage = market_value_safe / quantxi_equity_array # Menggunakan safe MV juga di sini

    dates_for_returns = dates[1:]
    dates_for_volatility = dates[WINDOW:]

    # ----------------------------------------------------------------------------------
    # Plot 3: Equity to Market Value Ratio
    # ----------------------------------------------------------------------------------
    # ax3.plot(dates, qg_equity_to_market_value * 100, label='QG Equity/MV (%)', color='magenta', linewidth=2)

    # ax3.axhline(0, color='black', linestyle='-', linewidth=1.5, label='0% (Bangkrut)')
    # ax3.axhline(25, color='orange', linestyle='--', linewidth=1.0, label='25% (Margin Minimum)')
    # ax3.axhline(100, color='green', linestyle='-', linewidth=1.5, alpha=0.8, label='100% (Tanpa Utang)')
    # ax3.set_title('Equity to Market Value Ratio (Modal vs Total Aset)', fontsize=14)
    # ax3.set_ylabel('Rasio E/MV (%)', fontsize=10)
    # ax3.tick_params(axis='y', labelcolor='black')
    # ax3.grid(True, linestyle='--', alpha=0.6)
    # ax3.legend(loc='best')
    # ax3.text(0.0, -0.2,
    #         'Catatan: Rasio modal (Equity) terhadap total aset (MV). Nilai < 100% menunjukkan penggunaan leverage.\nRasio modal (Equity) terhadap total aset (MV). Nilai < 100% menunjukkan penggunaan leverage.\nRasio modal (Equity) terhadap total aset (MV). Nilai < 100% menunjukkan penggunaan leverage.',
    #         transform=ax3.transAxes, fontsize=9, color='black', ha='left')

    # ----------------------------------------------------------------------------------
    ## Plot 8: Drawdown Kumulatif (Dipindah dari ax7)
    # ----------------------------------------------------------------------------------
    # qg_cumulative_return = np.insert(quantxi_equity_array / initial_eq, 0, 1.0)
    # qg_peak = np.maximum.accumulate(qg_cumulative_return)
    # qg_drawdown_plot = (qg_peak - qg_cumulative_return) / qg_peak

    # bh_cumulative_return = np.insert(buyhold_equity_array / initial_eq, 0, 1.0)
    # bh_peak = np.maximum.accumulate(bh_cumulative_return)
    # bh_drawdown_plot = (bh_peak - bh_cumulative_return) / bh_peak

    # plot_dates = np.insert(dates.values, 0, dates[0] - pd.Timedelta(days=1))

    # ax8.plot(plot_dates, qg_drawdown_plot * 100, label='QG Drawdown', color='blue')
    # ax8.plot(plot_dates, bh_drawdown_plot * 100, label='B&H Drawdown', color='red')
    # ax8.fill_between(plot_dates, qg_drawdown_plot * 100, color='blue', alpha=0.1)
    # ax8.fill_between(plot_dates, bh_drawdown_plot * 100, color='red', alpha=0.1)
    # ax8.set_title('Drawdown Kumulatif', fontsize=14)
    # ax8.set_xlabel('Tanggal', fontsize=10)
    # ax8.set_ylabel('Drawdown (%)', fontsize=10)
    # ax8.legend()
    # ax8.grid(True, linestyle='--', alpha=0.6)
    # ax8.text(0.0, -0.45,
    #         'Catatan: Kerugian dari puncak tertinggi (peak) yang dialami strategi. Semakin kecil dan pendek durasi, semakin baik.\nKerugian dari puncak tertinggi (peak) yang dialami strategi. Semakin kecil dan pendek durasi, semakin baik.\nKerugian dari puncak tertinggi (peak) yang dialami strategi. Semakin kecil dan pendek durasi, semakin baik.',
    #         transform=ax8.transAxes, fontsize=9, color='black', ha='left')

    # plt.close(fig)

    # pie_data = pd.DataFrame({
    #     'category': ['Elektronik', 'Pakaian', 'Makanan', 'Buku', 'Olahraga'],
    #     'value': [25, 30, 20, 15, 10]
    # })
    # --- 1. Data Preparation (Equity & Risk) ---
    quantxi_equity_df = pd.DataFrame(quantxi_equity_array, columns=['equity_value'])
    buyhold_equity_df = pd.DataFrame(buyhold_equity_array, columns=['buyhold_value'])
    market_value_df = pd.DataFrame(market_value_array, columns=['market_value'])

    # Penanganan pembagian nol
    market_val_safe = market_value_df['market_value'].replace(0, 1)
    ratio_solvability = quantxi_equity_df['equity_value'].values / market_val_safe.values

    # Drawdown Calculation
    qg_cumulative_return = quantxi_equity_array / initial_eq
    qg_peak = np.maximum.accumulate(qg_cumulative_return)
    qg_drawdown_plot = (qg_peak - qg_cumulative_return) / qg_peak

    bh_cumulative_return = buyhold_equity_array / initial_eq
    bh_peak = np.maximum.accumulate(bh_cumulative_return)
    bh_drawdown_plot = (bh_peak - bh_cumulative_return) / bh_peak

    # Sinkronisasi Sumbu Row 2
    qg_plot_sync = 1 - qg_drawdown_plot
    bh_plot_sync = 1 - bh_drawdown_plot

    # Contoh Daftar Saham dengan format lengkap
    daftar_saham = [
        f"{s} - IDX - Indonesia" for s in [
            'BBCA-Bank Central Asia', 'BBRI-Bank Rakyat Indoensia', 'BMRI-Bank Mandiri', 'TLKM-PT. Telkom', 'ASII-PT. Astra Indonesia', 'GOTO-PT. Gojek Indonesia', 'UNVR', 'ICBP', 'BBNI', 'ADRO',
            'PGAS', 'UNTR', 'KLBF', 'CPIN', 'BRPT', 'AMRT', 'INKP', 'TPIA', 'MDKA', 'ANTM',
            'ITMG', 'PTBA', 'SMGR', 'GGRM', 'HMSP', 'EXCL', 'ISAT', 'BUKA', 'HRUM', 'AKRA'
        ]
    ]

    # --- 3. Create Subplots (3 Rows) ---
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            '<b>Perbandingan Equity Curve: Quantxi vs Buy & Hold</b>',
            '<b>Anti Fragile (Solvability Ratio vs Drawdown)</b>',
            '<b>Rolling Sharpe Ratio Analysis</b>'
        ),
        vertical_spacing=0.07,
        specs=[[{"secondary_y": False}], [{"secondary_y": True}], [{"secondary_y": False}]]
    )

    # Row 1: Equity
    fig.add_trace(go.Scatter(y=quantxi_equity_df['equity_value'], name='Equity Quantxi', line=dict(color='blue'), legend='legend1'), row=1, col=1)
    fig.add_trace(go.Scatter(y=buyhold_equity_df['buyhold_value'], name='Buy & Hold', line=dict(color='red'), legend='legend1'), row=1, col=1)

    # Row 2: Anti Fragile
    fig.add_trace(go.Scatter(y=[1.0]*len(ratio_solvability), line=dict(width=0), showlegend=False, hoverinfo='skip'), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(y=ratio_solvability, name='DD Ratio (Q/BH)', line=dict(color='orange', width=3), fill='tonexty', fillcolor='rgba(255, 165, 0, 0.2)', legend='legend2'), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(y=qg_plot_sync, name='DD Quantxi', line=dict(color='green', width=1.5), legend='legend2'), row=2, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(y=bh_plot_sync, name='DD Buy & Hold', line=dict(color='yellow', width=1.5), legend='legend2'), row=2, col=1, secondary_y=True)

    # Row 3: Sharpe Ratio
    fig.add_trace(go.Scatter(y=qg_rolling_sharpe, name='QG Rolling Sharpe', line=dict(color='purple'), legend='legend3'), row=3, col=1)
    fig.add_trace(go.Scatter(y=bh_rolling_sharpe, name='BH Rolling Sharpe', line=dict(color='green'), legend='legend3'), row=3, col=1)

    # Konfigurasi Sumbu Row 2
    fig.update_yaxes(range=[0, 1], tickvals=[0, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0], showgrid=True, gridcolor='LightGrey', row=2, col=1, secondary_y=False)
    fig.update_yaxes(range=[0, 1], tickvals=[0, 0.2, 0.4, 0.6, 0.7, 0.8, 1.0], ticktext=["100%", "80%", "60%", "40%", "30%", "20%", "0%"], showgrid=False, row=2, col=1, secondary_y=True)
    fig.add_hline(y=0.3, line_dash="dot", line_color="red", line_width=2, row=2, col=1)

    # --- 4. Performance Comparison Table ---
    # Kalkulasi data akhir
    final_qg_return = ((quantxi_equity_array[-1] / initial_eq) - 1) * 100
    final_bh_return = ((buyhold_equity_array[-1] / initial_eq) - 1) * 100
    max_drawdown_qg = np.max(qg_drawdown_plot) * 100
    max_drawdown_bh = np.max(bh_drawdown_plot) * 100

    data_perf = {
        "Metrik Performa": ["Final Return (%)", "Max Drawdown (%)", "Final Equity Value", "Sharpe Ratio (Avg)"],
        "Sistem AI (Quantxi)": [f"{final_qg_return:.2f}%", f"{max_drawdown_qg:.2f}%", f"{quantxi_equity_array[-1]:,.0f}", f"{np.mean(qg_rolling_sharpe):.2f}"],
        "Buy & Hold": [f"{final_bh_return:.2f}%", f"{max_drawdown_bh:.2f}%", f"{buyhold_equity_array[-1]:,.0f}", f"{np.mean(bh_rolling_sharpe):.2f}"]
    }
    df_perf = pd.DataFrame(data_perf)

    # Mengubah tabel menjadi HTML agar terlihat cantik
    # table_html = f"""
    # <div style="margin-top: 30px;">
    #     <h3 style="color: #2c3e50;">📊 Perbandingan Performa Akhir</h3>
    #     {df_perf.to_html(index=False, classes='table table-striped', justify='center')}
    # </div>
    # <style>
    #     .table {{ width: 100%; border-collapse: collapse; font-family: sans-serif; }}
    #     .table th {{ background-color: #2c3e50; color: white; padding: 12px; }}
    #     .table td {{ border: 1px solid #dee2e6; padding: 10px; text-align: center; }}
    #     .table tr:nth-child(even) {{ background-color: #f2f2f2; }}
    # </style>
    # """

    # --- 5. Display Everything ---
    fig.update_layout(width=1200, height=1400, margin=dict(l=100, r=100, t=80, b=80))

    # Menampilkan berurutan
    display_full_report_new_tab(fig, daftar_saham, df_perf)
    # display(HTML(portfolio_html)) # 1. Daftar Saham
    # open_new_tab_base64(fig)               # 2. Tiga Grafik
    # display(HTML(table_html))     # 3. Tabel Perbandingan


    start_button.disabled = False
    reset_button.disabled = False
    # progress.visible=False

error_pane = pn.pane.Alert("No data available!", alert_type="warning", width=1115, visible=False)

def on_start_click(b):
    if len(test_data) == 0:
      error_pane.visible = True
      return
    else:
      start_simulation(test_data, dataHarga, tanggal)

start_button.on_click(on_start_click)

footer_text1 = pn.pane.HTML("""<div style='text-align: center;line-height: 1.35; font-size: 11.5px'>I am commitment for realistic trading simulation also open source and transparant testing to prove that QuantGenius actually works. Feel free to modify the code to suit your specific testing methodology. You also can test using jupyter notebook or google colab and kaggle or you can create your own testing system by using our api docs. I am commitment for realistic trading simulation also open source and transparant testing to prove that QuantGenius actually works. Feel free to modify the code to suit your specific testing methodology. You also can test using jupyter notebook or google colab and kaggle or you can create your own testing system by using our api docs. Check details in our Github.</div>""", sizing_mode='fixed', styles={'width': '95%', 'margin-left': '10px'})
main = pn.Column(centered_logo, title_text, main_accordion, start_button, error_pane, progress_pane, footer_text1, centered_badges, sizing_mode='stretch_width')
main.show()

Launching server at http://localhost:42131


0.9777388981755539
0.9890983020386217
